# Gerar pela API ou conferir uma proposta

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="02_teste_inferencia_planilha_turmas"


{'mode': 'READ_EXISTING',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': '8a186e1f-3eb9-4819-b6ec-f99ff331853e'}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Preflight e seleção. READ_EXISTING não escreve; GENERATE_AND_VALIDATE envia POST ao worker. Escolha explicitamente snapshot e modelo/versão; confirme a escrita e defina request_key.

In [2]:
from src.workbench.contract_review import preflight,candidates,fetch_payload,reconcile
display(pd.DataFrame(preflight(cfg)));display(pd.DataFrame(candidates(cfg)))
proposal_id=str(params.proposal_id) if params.proposal_id else ""


,dependency,status
0,API/OpenAPI/auth negative/MLflow SDK,PASS


,execution_id,generated_at,model_name,model_version,status,row_count
0,3c047abe-c2f3-44df-ad4d-157a8d61feb8,2026-11-04 20:12:09.721612+00:00,hi.demo.integration.5a00d837,1,DRAFT,90
1,8ba35d3e-70ee-4719-943a-4b607d2549ee,2026-11-04 19:29:06.312865+00:00,planilha-turmas-demo,15,DRAFT,90
2,896ad0e3-1a39-4c77-890b-0c622aad76f3,2026-11-04 19:24:44.231085+00:00,hi.demo.integration.d0b9bf81,1,DRAFT,90
3,6391c652-a079-4a73-a6c6-a385e8b9774f,2026-11-04 19:24:32.596731+00:00,hi.demo.probe.168e4337,1,DRAFT,90
4,712b11cd-a96a-4bb9-83bb-691724ceae88,2026-11-04 19:24:24.629371+00:00,planilha-turmas-demo,15,DRAFT,90
5,3d318979-6f71-4fad-bb8d-a89bada3e7d0,2026-11-04 18:58:39.197445+00:00,hi.demo.integration.0133d031,1,DRAFT,90
6,9c95f92a-87c7-4c86-8922-27fda3942b1d,2026-11-04 18:41:30.430300+00:00,planilha-turmas-demo,14,DRAFT,90
7,b2ad0068-b2c3-489e-94c0-c69a0b82224e,2026-11-04 18:13:30.041129+00:00,hi.demo.integration.760099da,1,DRAFT,90
8,b6b8b14b-33a1-4de1-bc01-ca839afdde2d,2026-11-04 18:13:18.591249+00:00,hi.demo.probe.fa61f2fa,1,DRAFT,90
9,2f6e1b6d-07f8-40a7-812d-df1b9228ea72,2026-11-04 18:03:18.059513+00:00,planilha-turmas-demo,14,DRAFT,90


2. Submeter geração somente no modo opt-in. Nenhuma persistência é feita pelo notebook. Esperado: job, correlation_id e status_url.

In [3]:
if params.mode=="GENERATE_AND_VALIDATE":
    params.authorize()
    if not params.snapshot_id or not params.model_version:
        raise ValueError("Selecione snapshot_id e model_version explicitamente")
    from src.workbench.operations_sdk import generate
    payload=dict(cycle_code=cfg.cycle,snapshot_id=str(params.snapshot_id),
        model_selectors=[dict(model_name=params.model_name,model_version=params.model_version)],
        capacity=params.capacity,requested_by=params.requester,confirm_demo=True)
    display(payload)
    submitted=generate(cfg,payload,params.request_key)
    display(submitted)
    finished=await_job(cfg,submitted["status_url"])
    if finished["status"]!="COMPLETED":
        raise RuntimeError("Geração não completou; consulte o job persistido: "+str(finished))
    proposal_id=finished["proposal_id"]
    evidence.update(request=payload,job=finished)
if not proposal_id:
    raise ValueError("Selecione proposal_id da tabela ou configure modo GENERATE_AND_VALIDATE")


3. Obter payload integral paginado e validar schema/hash. Este é o JSON consumido pela aplicação.

In [4]:
pages,offers=fetch_payload(cfg,proposal_id,page_size=50)
for page in pages: display(page)
display(pd.json_normalize(offers))
display(pd.DataFrame(pages[0]["validation"]))
assert not any(r["status"]=="FAIL" for r in pages[0]["validation"])
evidence.update(proposal_id=proposal_id,http_pages=pages)


{'schema_version': 'poc-planilha-turmas-v1',
 'contract_status': 'PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'proposal_revision': 1,
 'academic_cycle': {'academic_cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'cycle_code': '2026-1'},
 'status': 'DRAFT',
 'source_type': 'DEMO',
 'generated_at_utc': '2026-11-04T19:24:24.629371Z',
 'lineage': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'training_campaign_id': None,
  'ruleset_version': 'POC-whole-cohort-ceil-capacity-v1',
  'model_usages': [{'usage_id': '6ea71abd-f407-405e-8854-f94d97d6758c',
    'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
    'inference_id': 'be5cf3b8-2839-4658-bc50-e8d24d4c2add',
    'model_key': 'planilha-

{'schema_version': 'poc-planilha-turmas-v1',
 'contract_status': 'PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'proposal_revision': 1,
 'academic_cycle': {'academic_cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'cycle_code': '2026-1'},
 'status': 'DRAFT',
 'source_type': 'DEMO',
 'generated_at_utc': '2026-11-04T19:24:24.629371Z',
 'lineage': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'training_campaign_id': None,
  'ruleset_version': 'POC-whole-cohort-ceil-capacity-v1',
  'model_usages': [{'usage_id': '6ea71abd-f407-405e-8854-f94d97d6758c',
    'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
    'inference_id': 'be5cf3b8-2839-4658-bc50-e8d24d4c2add',
    'model_key': 'planilha-

,offer_id,campus_id,course_id,period_id,discipline_id,cohort,shift,offer_type,status,forecast_raw,...,students.regular,students.dependency,students.elective,students.total,students.origins,modulation.rule_version,modulation.capacity,modulation.proposed_groups,modulation.groups,modulation.provisional
0,3338dee0-4d05-5c12-85a1-433633a18d50,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
1,5778aa1a-d3eb-5991-a62e-69c80ff97456,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
2,12700109-403e-5d63-9e9b-54c8b387f44d,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
3,f8152078-0673-538e-8b14-1ab65ffa6b79,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,None,None,None,SUGERIDA,103.245511,...,None,None,None,104,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
4,1ee59f8d-e1d0-5a9b-9b83-a7202eec5993,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,None,None,None,SUGERIDA,103.245511,...,None,None,None,104,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,b260c47c-7576-5bf6-95a1-2c9c510a0080,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,11e42499-2538-5286-a8b9-3fda55802247,d30d0d0a-6af3-5ba7-b60d-e25d687c1e69,None,None,None,SUGERIDA,96.586051,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
86,b87b5c43-49c7-5068-ab22-8de850545461,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,11e42499-2538-5286-a8b9-3fda55802247,e7f05bc1-407f-5670-84d7-aae409217d6f,None,None,None,SUGERIDA,96.586051,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
87,7cad49ef-26d6-5f2a-ba56-4647ae8fca0e,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,1494b22f-3999-5714-ba12-3e8a1c65117c,5ebf59a4-4457-529e-98cb-04d804233a7c,None,None,None,SUGERIDA,96.497162,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
88,0bec07bc-6a9d-5107-97bf-8f0b6cc77184,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,1494b22f-3999-5714-ba12-3e8a1c65117c,7064e2c5-ce43-55f0-b151-5d1dd109de0f,None,None,None,SUGERIDA,96.497162,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True


,rule_id,status,reason,evidence
0,POC-DRAFT,PASS,Proposta demonstrativa sem aprovação automática,proposal_generations.status
1,RF008-partial,PASS,Sem docente/horário/sala na representação inic...,offers.teacher_suggestion
2,RGN001/US027,NOT_EVALUATED,Decisão humana e prevalência de edições não ex...,DEMO scope
3,RGN002/RF010,NOT_EVALUATED,Integração institucional entre todos os cursos...,DEMO scope
4,RGN003/RF021–025/US029–030,NOT_EVALUATED,Alvo agregado; composição regular/dependência/...,DEMO scope
5,RGN023/US028,NOT_EVALUATED,Gate de validação da matriz depende da aplicação,DEMO scope
6,RF009/RF020/US026,NOT_EVALUATED,"Turno, coorte de ingresso e matriz homologada ...",DEMO scope
7,RF040–047/US033–034,NOT_EVALUATED,Capacidade DEMO não é regra institucional; uni...,DEMO scope
8,RF117–125,NOT_EVALUATED,Sem elegibilidade PRIME/Escola/turno ou aprova...,DEMO scope
9,RF155–159/US005–006,NOT_EVALUATED,SQL sintético e temporal; nenhuma integração i...,DEMO scope


4. Reconciliação somente leitura: linhas SQL, versões Registry, snapshot e bytes RustFS. Esperado: mesmos IDs, quantidades e hashes.

In [5]:
evidence["reconciliation"]=reconcile(cfg,proposal_id,pages,offers)
display(evidence["reconciliation"])
display(Markdown(f"[API](http://localhost:8000/v1/proposals/{proposal_id}/application-view) · [MLflow]({cfg.ui}) · [RustFS](http://localhost:9001)"))


{'sql': {'database': 'planejamento_db',
  'schema': 'planning',
  'status': 'PASS',
  'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
  'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
  'model_version': '15',
  'proposal_status': 'DRAFT',
  'stored_rows': 90,
  'proposed_students': 8694,
  'proposed_groups': 270,
  'query': 'sql/queries/verify_proposal.sql'},
 'storage': [{'bucket': 'pucpr-poc',
   'key': 'mlflow/4/models/m-342590e0c9ca494e99e6d746160225f8/artifacts/MLmodel',
   'bytes': 1336,
   'sha256': '752dfba474fc343bf7f6654ed174c08207af206086532c6ba0480b631622ccfc',
   'object_version_id': '9d81a473-df96-4333-9fe9-1e6ee9a5d1e1',
   'model_version': '15',
   'status': 'PASS'},
  {'bucket': 'pucpr-poc',
   'key': 'mlflow/4/models/m-342590e0c9ca494e99e6d746160225f8/artifacts/model.pkl',
   'bytes': 8671,
   'sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
   'object_version_id': 'f0e9a40e-7cc0-4ad3-af4d-a358027084a8',
   'model_version': '15',
   

[API](http://localhost:8000/v1/proposals/712b11cd-a96a-4bb9-83bb-691724ceae88/application-view) · [MLflow](http://127.0.0.1:5000) · [RustFS](http://localhost:9001)

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [6]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v2/02_teste_inferencia_planilha_turmas.json'